# Part 6 - XGBoost Competition Pipeline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

TARGET='addicted_label'
ID='id'

FEATURES=[c for c in train_fe.columns if c not in [ID,TARGET]]

X=train_fe[FEATURES].copy()
y=train_fe[TARGET]
X_test=test_fe[FEATURES].copy()

cat_cols=X.select_dtypes(include=['object','category']).columns.tolist()

# XGBoost requires numeric input
enc=OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

if len(cat_cols)>0:
    X[cat_cols]=enc.fit_transform(X[cat_cols].astype(str))
    X_test[cat_cols]=enc.transform(X_test[cat_cols].astype(str))


## Parameters

In [ ]:
params=dict(
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=4000,
    learning_rate=0.02,
    max_depth=7,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.2,
    reg_lambda=2.0,
    gamma=0.1,
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

oof=np.zeros(len(X))
pred=np.zeros(len(X_test))
scores=[]
fi=pd.DataFrame(index=FEATURES)


## Cross Validation

In [ ]:
for fold,(tr,va) in enumerate(cv.split(X,y),1):

    print("="*60)
    print("Fold",fold)

    model=XGBClassifier(**params)

    model.fit(
        X.iloc[tr],y.iloc[tr],
        eval_set=[(X.iloc[va],y.iloc[va])],
        verbose=200
    )

    val=model.predict_proba(X.iloc[va])[:,1]
    oof[va]=val

    auc=roc_auc_score(y.iloc[va],val)
    scores.append(auc)

    pred += model.predict_proba(X_test)[:,1]/cv.n_splits

    fi[f'fold_{fold}']=model.feature_importances_

    print(f"Fold {fold} AUC: {auc:.6f}")

print("\nMean CV:",np.mean(scores))
print("Overall OOF:",roc_auc_score(y,oof))


## Feature Importance

In [ ]:
fi["mean"]=fi.mean(axis=1)
fi=fi.sort_values("mean",ascending=False)

display(fi.head(20))

plt.figure(figsize=(8,8))
fi["mean"].head(20).sort_values().plot.barh()
plt.title("Top 20 XGBoost Features")
plt.tight_layout()
plt.show()


## Submission

In [ ]:
submission=sample.copy()
submission[TARGET]=pred
submission.to_csv("submission_xgboost.csv",index=False)

display(submission.head())
print("Saved submission_xgboost.csv")


### Next

Part 7 will focus on **Optuna hyperparameter tuning** for CatBoost, LightGBM, and XGBoost, saving the best parameters for the final ensemble.
